# Bloco 4 — Estacionariedade, cointegração e quebras estruturais

## Objetivo

Determinar a ordem de integração de cada série e investigar cointegração entre as 
variáveis para decidir a forma correta de especificação do modelo (VAR em níveis, 
VAR em diferenças, ou VECM).

## 1. Testes de estacionariedade — séries em nível

### Metodologia

Aplicamos dois testes complementares a cada série:

- **ADF (Augmented Dickey-Fuller)**: H₀ = série tem raiz unitária (não-estacionária)
- **KPSS (Kwiatkowski-Phillips-Schmidt-Shin)**: H₀ = série é estacionária

Os dois testes têm hipóteses nulas opostas, e ler ambos em conjunto reduz 
ambiguidade. Interpretação:

| ADF | KPSS | Conclusão |
|-----|------|-----------|
| Rejeita H₀ | Não rejeita H₀ | **Caso A**: série é I(0) (estacionária) |
| Não rejeita H₀ | Rejeita H₀ | **Caso B**: série é I(1) ou superior |
| Rejeita H₀ | Rejeita H₀ | **Caso C**: conflito (testar quebra estrutural) |
| Não rejeita H₀ | Não rejeita H₀ | **Caso D**: inconclusivo |

### Especificação determinística

A especificação da equação de teste depende do comportamento visual da série. 
Séries com tendência clara usam constante + tendência ("ct"); séries com média 
não-zero mas sem tendência usam apenas constante ("c").

| Série | Especificação | Justificativa |
|-------|--------------|---------------|
| `ln_ibcbr` | ct | Tendência crescente clara |
| `ln_cambio` | ct | Tendência crescente apesar de quebra em 2014-15 |
| `ln_commodities` | ct | Tendência crescente clara |
| `ln_m1` | ct | Tendência crescente forte |
| `ln_prod_industrial` | c | Estável global, mas com possível quebra em 2014-15 |
| `ln_credito_total` | ct | Tendência crescente forte |
| `ipca` | c | Variação % com média ≠ 0, sem tendência |
| `selic` | c | Taxa com nível variável mas sem tendência clara |
| `exp_ipca_12m` | c | Taxa com média ≠ 0, sem tendência |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

# Silencia warnings de KPSS quando p-valor estoura limites tabulados
warnings.filterwarnings('ignore', category=Warning, module='statsmodels')

DATA_PROCESSED = Path('../data/processed')

# Carrega o dataset tratado do Bloco 3
df = pd.read_csv(
    DATA_PROCESSED / 'series_tratadas.csv',
    index_col=0,
    parse_dates=True
)

# Separa séries macroeconômicas das dummies
series_macro = [c for c in df.columns if not c.startswith('d_')]
print(f"Séries a testar ({len(series_macro)}):")
for s in series_macro:
    print(f"  - {s}")

Séries a testar (9):
  - ln_ibcbr
  - ln_cambio
  - ln_commodities
  - ln_m1
  - ln_prod_industrial
  - ln_credito_total
  - ipca
  - selic
  - exp_ipca_12m


In [2]:
# Dicionário com especificação determinística de cada série
# 'c' = constante apenas | 'ct' = constante + tendência
especificacao = {
    'ln_ibcbr':           'ct',
    'ln_cambio':          'ct',
    'ln_commodities':     'ct',
    'ln_m1':              'ct',
    'ln_prod_industrial': 'c',
    'ln_credito_total':   'ct',
    'ipca':               'c',
    'selic':              'c',
    'exp_ipca_12m':       'c',
}

# Verifica que todas as séries têm especificação definida
faltantes = set(series_macro) - set(especificacao.keys())
if faltantes:
    print(f"⚠ Sem especificação: {faltantes}")
else:
    print("✓ Todas as séries têm especificação definida")

✓ Todas as séries têm especificação definida


In [3]:
def testar_estacionariedade(serie, nome, regression):
    """
    Aplica ADF e KPSS em uma série e retorna um dicionário consolidado.
    
    Parameters
    ----------
    serie : pd.Series
        Série temporal já limpa (sem NaN)
    nome : str
        Identificador da série
    regression : str
        Especificação determinística: 'c' (constante) ou 'ct' (constante + tendência)
    
    Returns
    -------
    dict com resultados dos dois testes e classificação do caso
    """
    serie_limpa = serie.dropna()
    
    # ADF
    adf_stat, adf_pvalue, adf_lags, adf_nobs, _, _ = adfuller(
        serie_limpa, regression=regression, autolag='AIC'
    )
    
    # KPSS — usa 'c' ou 'ct' também, parâmetro chamado 'regression' aqui também
    kpss_stat, kpss_pvalue, kpss_lags, _ = kpss(
        serie_limpa, regression=regression, nlags='auto'
    )
    
    # Classificação dos casos (a 5% de significância)
    adf_rejeita = adf_pvalue < 0.05  # rejeita raiz unitária → série estacionária
    kpss_rejeita = kpss_pvalue < 0.05  # rejeita estacionariedade → série não-estacionária
    
    if adf_rejeita and not kpss_rejeita:
        caso = 'A — Estacionária'
    elif not adf_rejeita and kpss_rejeita:
        caso = 'B — Não-estacionária'
    elif adf_rejeita and kpss_rejeita:
        caso = 'C — Conflito (testar quebra)'
    else:
        caso = 'D — Inconclusivo'
    
    return {
        'serie': nome,
        'spec': regression,
        'adf_stat': adf_stat,
        'adf_pvalue': adf_pvalue,
        'adf_lags': adf_lags,
        'kpss_stat': kpss_stat,
        'kpss_pvalue': kpss_pvalue,
        'kpss_lags': kpss_lags,
        'caso': caso,
    }

In [4]:
print("=" * 80)
print("TESTES DE ESTACIONARIEDADE — SÉRIES EM NÍVEL")
print("=" * 80)

resultados_nivel = []
for col in series_macro:
    spec = especificacao[col]
    res = testar_estacionariedade(df[col], col, spec)
    resultados_nivel.append(res)
    print(f"\n{col:25s} (spec: {spec})")
    print(f"  ADF:  stat={res['adf_stat']:>8.3f}  p-valor={res['adf_pvalue']:.4f}  lags={res['adf_lags']}")
    print(f"  KPSS: stat={res['kpss_stat']:>8.3f}  p-valor={res['kpss_pvalue']:.4f}  lags={res['kpss_lags']}")
    print(f"  → {res['caso']}")

TESTES DE ESTACIONARIEDADE — SÉRIES EM NÍVEL

ln_ibcbr                  (spec: ct)
  ADF:  stat=  -2.197  p-valor=0.4915  lags=2
  KPSS: stat=   0.448  p-valor=0.0100  lags=10
  → B — Não-estacionária

ln_cambio                 (spec: ct)
  ADF:  stat=  -3.060  p-valor=0.1160  lags=1
  KPSS: stat=   0.460  p-valor=0.0100  lags=10
  → B — Não-estacionária

ln_commodities            (spec: ct)
  ADF:  stat=  -2.757  p-valor=0.2133  lags=1
  KPSS: stat=   0.492  p-valor=0.0100  lags=10
  → B — Não-estacionária

ln_m1                     (spec: ct)
  ADF:  stat=  -2.373  p-valor=0.3939  lags=10
  KPSS: stat=   0.301  p-valor=0.0100  lags=10
  → B — Não-estacionária

ln_prod_industrial        (spec: c)
  ADF:  stat=  -2.659  p-valor=0.0814  lags=4
  KPSS: stat=   0.637  p-valor=0.0193  lags=10
  → B — Não-estacionária

ln_credito_total          (spec: ct)
  ADF:  stat=  -3.256  p-valor=0.0738  lags=14
  KPSS: stat=   0.572  p-valor=0.0100  lags=10
  → B — Não-estacionária

ipca             

C:\Users\lucas\AppData\Local\Temp\ipykernel_35472\2291843935.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, kpss_lags, _ = kpss(
C:\Users\lucas\AppData\Local\Temp\ipykernel_35472\2291843935.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, kpss_lags, _ = kpss(
C:\Users\lucas\AppData\Local\Temp\ipykernel_35472\2291843935.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_pvalue, kpss_lags, _ = kpss(
C:\Users\lucas\AppData\Local\Temp\ipykernel_35472\2291843935.py:26: InterpolationWarning: The test statistic is outside of the range of p-values available

In [5]:
# Converte lista de dicionários em DataFrame para visualização tabular
df_resultados_nivel = pd.DataFrame(resultados_nivel)

# Reorganiza colunas e arredonda
df_resultados_nivel = df_resultados_nivel[[
    'serie', 'spec', 'adf_stat', 'adf_pvalue', 'kpss_stat', 'kpss_pvalue', 'caso'
]].round({'adf_stat': 3, 'adf_pvalue': 4, 'kpss_stat': 3, 'kpss_pvalue': 4})

print("\nResumo dos testes em nível:\n")
print(df_resultados_nivel.to_string(index=False))


Resumo dos testes em nível:

             serie spec  adf_stat  adf_pvalue  kpss_stat  kpss_pvalue                 caso
          ln_ibcbr   ct    -2.197      0.4915      0.448       0.0100 B — Não-estacionária
         ln_cambio   ct    -3.060      0.1160      0.460       0.0100 B — Não-estacionária
    ln_commodities   ct    -2.757      0.2133      0.492       0.0100 B — Não-estacionária
             ln_m1   ct    -2.373      0.3939      0.301       0.0100 B — Não-estacionária
ln_prod_industrial    c    -2.659      0.0814      0.637       0.0193 B — Não-estacionária
  ln_credito_total   ct    -3.256      0.0738      0.572       0.0100 B — Não-estacionária
              ipca    c    -9.529      0.0000      0.116       0.1000     A — Estacionária
             selic    c    -1.781      0.3901      0.939       0.0100 B — Não-estacionária
      exp_ipca_12m    c    -2.481      0.1202      0.520       0.0371 B — Não-estacionária


## 2. Testes complementares

Os testes ADF e KPSS na seção anterior identificaram 8 séries em **Caso B** 
(não-estacionárias), com duas delas em zona fronteiriça:

- `ln_prod_industrial` (ADF p=0.0814)
- `ln_credito_total` (ADF p=0.0738)

Adicionalmente, três séries têm quebra estrutural visual:

- `selic` (mudança de regime monetário)
- `ln_cambio` (mudança em 2014-2015)
- `ln_prod_industrial` (mudança em 2014-2015)

Para investigar essas séries com mais rigor, aplicamos dois testes adicionais:

### 2.1 Phillips-Perron (PP)

Alternativa ao ADF com correção não-paramétrica para autocorrelação e 
heterocedasticidade. Não exige escolha de lags da regressão auxiliar, apenas 
janela para estimador de variância de longo prazo. Mesma H₀ que o ADF: 
série tem raiz unitária.

**Uso:** verificação cruzada das séries fronteiriças e robustez geral.

### 2.2 Zivot-Andrews

ADF modificado que permite **uma quebra estrutural endógena** (o teste escolhe 
a data ótima da quebra). Rejeitar H₀ significa que a série é estacionária 
em torno de uma quebra de nível e/ou tendência — ou seja, I(0) com quebra, 
não I(1).

**Uso:** séries com quebra visual identificada (`selic`, `ln_cambio`, 
`ln_prod_industrial`).

**Limitação:** assume apenas uma quebra. Para múltiplas, seria preciso 
Lee-Strazicich ou Bai-Perron, fora do escopo deste estudo.

In [6]:
from arch.unitroot import PhillipsPerron, ZivotAndrews

def teste_pp(serie, nome, trend='c'):
    """
    Aplica Phillips-Perron.
    
    trend: 'n' (sem const, sem trend), 'c' (constante), 'ct' (constante + tendência)
    """
    serie_limpa = serie.dropna()
    pp = PhillipsPerron(serie_limpa, trend=trend)
    return {
        'serie': nome,
        'spec': trend,
        'pp_stat': pp.stat,
        'pp_pvalue': pp.pvalue,
        'pp_lags': pp.lags,
        'rejeita_5pct': pp.pvalue < 0.05,
    }

print("=" * 80)
print("TESTES PHILLIPS-PERRON — SÉRIES EM NÍVEL")
print("=" * 80)

resultados_pp = []
for col in series_macro:
    spec = especificacao[col]
    res = teste_pp(df[col], col, trend=spec)
    resultados_pp.append(res)
    sinal = '✓ rejeita H₀' if res['rejeita_5pct'] else '✗ não rejeita'
    print(f"\n{col:25s} (spec: {spec})")
    print(f"  PP:   stat={res['pp_stat']:>8.3f}  p-valor={res['pp_pvalue']:.4f}  lags={res['pp_lags']}")
    print(f"  → {sinal}")

TESTES PHILLIPS-PERRON — SÉRIES EM NÍVEL

ln_ibcbr                  (spec: ct)
  PP:   stat=  -2.082  p-valor=0.5559  lags=16
  → ✗ não rejeita

ln_cambio                 (spec: ct)
  PP:   stat=  -2.715  p-valor=0.2301  lags=16
  → ✗ não rejeita

ln_commodities            (spec: ct)
  PP:   stat=  -2.334  p-valor=0.4151  lags=16
  → ✗ não rejeita

ln_m1                     (spec: ct)
  PP:   stat=  -1.645  p-valor=0.7743  lags=16
  → ✗ não rejeita

ln_prod_industrial        (spec: c)
  PP:   stat=  -3.258  p-valor=0.0169  lags=16
  → ✓ rejeita H₀

ln_credito_total          (spec: ct)
  PP:   stat=  -1.268  p-valor=0.8957  lags=16
  → ✗ não rejeita

ipca                      (spec: c)
  PP:   stat=  -9.686  p-valor=0.0000  lags=16
  → ✓ rejeita H₀

selic                     (spec: c)
  PP:   stat=  -2.918  p-valor=0.0433  lags=16
  → ✓ rejeita H₀

exp_ipca_12m              (spec: c)
  PP:   stat=  -5.575  p-valor=0.0000  lags=16
  → ✓ rejeita H₀


In [7]:
# Combina resultados ADF (nivel) e PP em tabela única para comparação
df_pp = pd.DataFrame(resultados_pp)
df_comp = df_resultados_nivel[['serie', 'spec', 'adf_pvalue', 'kpss_pvalue', 'caso']].merge(
    df_pp[['serie', 'pp_pvalue']],
    on='serie'
)
df_comp = df_comp[['serie', 'spec', 'adf_pvalue', 'pp_pvalue', 'kpss_pvalue', 'caso']]
df_comp = df_comp.round(4)

print("\nComparação ADF vs PP:\n")
print(df_comp.to_string(index=False))


Comparação ADF vs PP:

             serie spec  adf_pvalue  pp_pvalue  kpss_pvalue                 caso
          ln_ibcbr   ct      0.4915     0.5559       0.0100 B — Não-estacionária
         ln_cambio   ct      0.1160     0.2301       0.0100 B — Não-estacionária
    ln_commodities   ct      0.2133     0.4151       0.0100 B — Não-estacionária
             ln_m1   ct      0.3939     0.7743       0.0100 B — Não-estacionária
ln_prod_industrial    c      0.0814     0.0169       0.0193 B — Não-estacionária
  ln_credito_total   ct      0.0738     0.8957       0.0100 B — Não-estacionária
              ipca    c      0.0000     0.0000       0.1000     A — Estacionária
             selic    c      0.3901     0.0433       0.0100 B — Não-estacionária
      exp_ipca_12m    c      0.1202     0.0000       0.0371 B — Não-estacionária


In [8]:
def teste_za(serie, nome, trend='c'):
    """
    Aplica Zivot-Andrews permitindo uma quebra endógena.
    
    trend: 'c' (quebra no intercepto), 't' (quebra na tendência), 
           'ct' (quebra em ambos)
    """
    serie_limpa = serie.dropna()
    za = ZivotAndrews(serie_limpa, trend=trend)
    return {
        'serie': nome,
        'spec': trend,
        'za_stat': za.stat,
        'za_pvalue': za.pvalue,
        'za_lags': za.lags,
        'rejeita_5pct': za.pvalue < 0.05,
    }

# Aplicar apenas nas séries com suspeita de quebra
series_com_quebra = {
    'selic': 'c',           # quebra de nível esperada
    'ln_cambio': 'ct',      # quebra de nível e tendência
    'ln_prod_industrial': 'ct',  # quebra de nível e tendência
}

print("=" * 80)
print("TESTES ZIVOT-ANDREWS — SÉRIES COM SUSPEITA DE QUEBRA")
print("=" * 80)

resultados_za = []
for col, spec in series_com_quebra.items():
    res = teste_za(df[col], col, trend=spec)
    resultados_za.append(res)
    sinal = '✓ rejeita H₀ (I(0) com quebra)' if res['rejeita_5pct'] else '✗ não rejeita (I(1))'
    print(f"\n{col:25s} (spec: {spec})")
    print(f"  ZA:   stat={res['za_stat']:>8.3f}  p-valor={res['za_pvalue']:.4f}  lags={res['za_lags']}")
    print(f"  → {sinal}")

TESTES ZIVOT-ANDREWS — SÉRIES COM SUSPEITA DE QUEBRA

selic                     (spec: c)
  ZA:   stat=  -3.550  p-valor=0.6474  lags=9
  → ✗ não rejeita (I(1))

ln_cambio                 (spec: ct)
  ZA:   stat=  -3.876  p-valor=0.5869  lags=1
  → ✗ não rejeita (I(1))

ln_prod_industrial        (spec: ct)
  ZA:   stat=  -4.734  p-valor=0.1261  lags=4
  → ✗ não rejeita (I(1))


## 2.3 Síntese da Etapa 2 — Testes Complementares

### Phillips-Perron — divergências em relação ao ADF

O teste PP confirmou o ADF para 6 das 9 séries, mas divergiu em 3 casos:

| Série | ADF p-valor | PP p-valor | Divergência |
|-------|:-----------:|:----------:|:-----------:|
| `ln_prod_industrial` | 0.0814 | 0.0169 | PP rejeita H₀, ADF marginalmente não |
| `selic` | 0.3901 | 0.0433 | PP rejeita H₀, ADF firmemente não |
| `exp_ipca_12m` | 0.1202 | 0.0000 | PP rejeita H₀ fortemente, ADF não |

### Interpretação das divergências

**`ln_prod_industrial`**: a heterocedasticidade entre regimes pré e pós-2014 
explica a divergência. PP é mais robusto a heterocedasticidade que ADF. 
Combinado com KPSS rejeitando estacionariedade (p=0.0193), configura **Caso C** 
(conflito entre testes), tipicamente associado a má especificação por quebra 
estrutural não-modelada.

**`selic`**: divergência mais expressiva. ADF interpreta as oscilações dos 
ciclos de aperto/afrouxamento monetário como evidência de raiz unitária; PP 
com janela larga de Newey-West (16 lags) absorve parte dessas oscilações na 
correção de variância. Compatível com hipótese de série I(0) em torno de 
regimes que mudam, mas o ADF assume regime único.

**`exp_ipca_12m`**: outlier severo no início da série (2002-2004, crise 
pré-Lula) provavelmente puxa o ADF para não-rejeição. PP "vê através" do 
outlier por construção. A 5% temos uma evidência mista (Caso D no par 
ADF+KPSS, mas PP rejeita), justificando análise complementar.

### Limitação técnica do PP

A escolha automática de lags pela regra de Schwert resultou em **janela de 
16 lags** para todas as séries. Janela larga garante correção robusta a 
autocorrelação distante, mas é conhecida por induzir baixo poder em amostras 
finitas e distorção de tamanho. Por essa razão, **PP é tratado como teste 
complementar, não substituto do ADF**.

### Zivot-Andrews — quebra estrutural endógena

O teste foi aplicado às três séries com quebra visual identificada na 
inspeção do Bloco 3:

| Série | Especificação | Estatística | p-valor | Conclusão |
|-------|:------------:|:-----------:|:-------:|:---------:|
| `selic` | c | -3.550 | 0.6474 | Não rejeita |
| `ln_cambio` | ct | -3.876 | 0.5869 | Não rejeita |
| `ln_prod_industrial` | ct | -4.734 | 0.1261 | Não rejeita |

Nenhuma das três séries permite rejeição da hipótese nula de raiz unitária 
em torno de uma quebra. Este resultado **contradiz a inspeção visual** 
(quebras claras em câmbio 2014-15, produção industrial 2014-15, e mudança 
de regime na Selic) e a evidência histórica brasileira.

### Por que ZA pode falhar mesmo com quebra real

1. **ZA assume uma única quebra.** Selic e câmbio apresentam múltiplas 
   quebras candidatas (2008, 2014, 2020), e o teste fica indeciso entre 
   datas, perdendo poder estatístico.

2. **Poder estatístico modesto.** O teste exige que a quebra esteja bem 
   identificada em torno de uma data específica para gerar rejeição.

3. **Especificação "ct" permissiva.** Permitir quebra tanto em nível quanto 
   em tendência torna o teste menos potente.

4. **Seleção automática de lags** (regra Ng-Perron) pode ter encerrado o 
   procedimento muito cedo, principalmente em `ln_cambio` (lags=1).

### Decisão metodológica

Adotamos a **abordagem de precaução**: séries com evidência conflitante 
entre testes são tratadas como **I(1)** para fins de modelagem, mas 
**dummies de quebra estrutural** serão incluídas como variáveis exógenas 
no VAR/VECM para capturar o efeito das mudanças de regime identificadas 
visualmente.

**Datas de quebra propostas** (a refinar no Bloco 5):
- `selic`: 2017-01 (início do ciclo de juros baixos sob Ilan Goldfajn)
- `ln_cambio`: 2014-09 (fim do superciclo de commodities + mudança política)
- `ln_prod_industrial`: 2014-09 (início da "década perdida" da indústria)

### Tabela consolidada — ordem de integração preliminar

| Série | ADF | PP | KPSS | ZA | Decisão |
|-------|:---:|:---:|:----:|:---:|:-------:|
| `ln_ibcbr` | I(1) | I(1) | I(1) | — | **I(1)** |
| `ln_cambio` | I(1) | I(1) | I(1) | I(1) | **I(1)** |
| `ln_commodities` | I(1) | I(1) | I(1) | — | **I(1)** |
| `ln_m1` | I(1) | I(1) | I(1) | — | **I(1)** |
| `ln_prod_industrial` | I(1)? | I(0)? | I(1) | I(1) | **I(1)** + dummy quebra |
| `ln_credito_total` | I(1) | I(1) | I(1) | — | **I(1)** |
| `ipca` | I(0) | I(0) | I(0) | — | **I(0)** |
| `selic` | I(1) | I(0)? | I(1) | I(1) | **I(1)** + dummy quebra |
| `exp_ipca_12m` | I(1)? | I(0)? | I(1)? | — | **I(1)** (precaução) |

### Próximo passo

Testar primeira diferença das 8 séries identificadas como I(1) para confirmar 
que diferenciar uma vez basta — ou seja, que $\Delta y_t \sim I(0)$. Se 
alguma diferença permanecer não-estacionária, a série seria I(2), o que 
exigiria tratamento especial.